# V-Energy Grid 360 | Gold — Fila de perdas não técnicas (entrega final)

**Notebook principal:** `gold_fila_perdas` · **Versão:** V10 · **Responsável:** Artur · **Camada:** Gold · **Fonte:** cinco tabelas Silver.

## Objetivo de negócio
Oferecer ao Rafael uma lista ordenada de UCs com motivo auditável, pontuação, indicação de kWh potencial e **ação recomendada**. A capacidade de 200 inspeções/mês é um limite máximo — não justifica fabricar candidatos nem tratar consumo atípico como fraude confirmada.

## Decisão operacional definitiva
A versão V8 (executada em 23/10/2026) tinha **178 elegíveis**; a V9 os reduziu a 8 ao excluir estimadas com queda abaixo do limiar. A V10 **preserva os critérios de elegibilidade que produziram a V8** e faz uma mudança mais útil: divide os elegíveis em duas ações na **mesma fila**:

1. **`INSPECAO_PRIORITARIA` (prioridade 1):** queda sazonal/absoluta qualificada, independente das estimadas. Recomendar análise da equipe de perdas e eventual ordem de campo.
2. **`VALIDACAO_DOCUMENTAL` (prioridade 2):** elegível pelo score e por leituras estimadas recorrentes, mas sem queda qualificada. Conferir leitura, faturamento ou histórico antes de autorizar ordem de inspeção.

A auditoria continua com as 45.000 UCs e status de retenção, os motivos e todos os metadados de execução. **Nem toda linha da fila é ordem de inspeção**: o portal deve observar `acao_recomendada`.

## Decisões preservadas
- Reunião de 21/10: negativos inválidos preservados na Silver e excluídos dos cálculos Gold.
- A1: até 1 kWh de arredondamento é informativo; divergência material é reconciliada antes do cálculo.
- A2: inspeções nos 12 meses anteriores em revisão, com resultado da inspeção disponível.
- A3: 30 pontos por queda absoluta só quando a queda percentual é ≥20% e a absoluta é ≥120 kWh/mês.
- A4: versões Delta de todas as fontes, `id_execucao` e controle de origem imutável durante a execução.
- A5: troca de medidor desde o início da janela-base até o corte é atenuante/retida para revisão.

**Dados confidenciais:** não commitar outputs nem arquivos com IDs, CPF, endereços e amostras.


## Contrato de saída e faixas de ação

| Campo | Definição |
|---|---|
| `status_triagem` | Elegibilidade e justificativas de retenção da auditoria |
| `acao_recomendada` | `INSPECAO_PRIORITARIA`, `VALIDACAO_DOCUMENTAL` ou `NAO_APLICAVEL` |
| `prioridade_operacional` | 1 = campo prioritário; 2 = validação documental; 9 = não elegível |
| `queda_qualificada` | Motivo `QUEDA_SAZONAL_RELEVANTE` ou `QUEDA_ABSOLUTA_RELEVANTE` |
| `score`, `p_queda_pct`, `p_queda_kwh`, `p_estimada` | Pontuação e decomposição, **não é probabilidade de fraude** |
| `kwh_estimado` | Proxy de diferença sazonal, **não energia recuperada comprovada** |
| `motivos_json`, `fatores_atenuantes_json` | Motivos e atenuantes serializados para exportação |
| `ultima_inspecao`, `resultado_ultima_inspecao` | Evidência para decidir sobre revisitas |
| `versao_regra`, `versoes_silver_json`, `id_execucao` | Linhagem e consistência entre as duas tabelas Gold |

**Consumo pela API:** para exibir todas as oportunidades, consultar `tb_fila_perdas` em `posicao_fila`; para gerar ordem de campo sem revisão documental, filtrar `acao_recomendada = 'INSPECAO_PRIORITARIA'`. Outros casos necessitam confirmação humana antes de gerar ordem.

## Metodologia e parâmetros
Janelas de três meses comparadas ao mesmo trimestre do ano anterior; ≥2 pares válidos, ≥1 observado; score mínimo 20; percentil 99,9 para outliers por classe quando a flag não existir. Limiares percentuais e pesos são heurísticos operacionais, **não normas da ANEEL**. Geração distribuída, troca de medidor, inconsistência de leitura e inspeções recentes são controladas como explicações alternativas.


## Execução e publicação V10

Configure os valores efetivos e execute **Run All**, nunca apenas a célula de escrita:
- Validação: `modo_exploracao=false`, `publicar_gold=false`.
- Publicação direta ou Job: `catalog=workspace`, `modo_exploracao=false`,
  `publicar_gold=true`, `homologacao_regras=true`, `versoes_silver_fixas_json` vazio.
- A exploração passa explicitamente `modo_exploracao=true`, `publicar_gold=false`
  e `homologacao_regras=false`; não integra o Workflow.

Os defaults não comprovam os valores de runtime. O início e a pré-publicação imprimem
todos os parâmetros efetivos e o modo resolvido. Parâmetros alterados entre as etapas
interrompem a escrita. A origem direta/%run/Job deve ser conferida no histórico do Databricks.

Use um único publicador: `max_concurrent_runs=1` no Job e nenhuma execução manual
concorrente. As verificações de HEAD detectam alterações, mas não são um lock entre tabelas.
Consumidores só devem ser liberados após a dupla conferência; consultar `EXECUCAO_V10.md`.
Os totais 178/8/170 são referências históricas fornecidas, nunca metas fixadas no código.


In [0]:
from datetime import timedelta, datetime, timezone
from pyspark.sql import functions as F

VERSAO_REGRA = "heuristica_v10_fila_operacional_duas_etapas"
INSTANTE_EXECUCAO_UTC = datetime.now(timezone.utc)
ID_EXECUCAO = INSTANTE_EXECUCAO_UTC.strftime("%Y%m%dT%H%M%S%fZ")
JANELA_MESES = 3
MIN_PARES = 2
MIN_PARES_OBSERVADOS = 1
LIMIAR_PROP_PARES_ESTIMADOS = 2.0 / 3.0
LIMIAR_QUEDA_PCT = 0.20
LIMIAR_QUEDA_KWH_MES = 50.0
LIMIAR_ABS_ELEVADO_KWH_MES = 120.0
MIN_SCORE_ELEGIVEL = 20.0
MESES_INSPECAO_RECENTE = 12
# A3: queda absoluta só pontua e vira motivo quando a variação percentual é relevante.
# Limiar de 20% é proposta de negócio, não parâmetro regulatório; requer homologação.
LIMIAR_PCT_PARA_QUEDA_ABSOLUTA = LIMIAR_QUEDA_PCT
TOP_N = 200
PERCENTIL_OUTLIER_CLASSE = 0.999
PRECISAO_PERCENTIL_OUTLIER = 10000
MIN_COBERTURA_ULTIMO_MES = 0.80  # defesa contra competência parcialmente carregada
# Padrão real da Silver: avisar e bloquear se flags essenciais desaparecerem.
FLAGS_OBRIGATORIAS = (
    "flag_leitura_estimada", "flag_primeira_leitura_pos_troca",
    "flag_consumo_negativo", "flag_consumo_inconsistente",
    "flag_leitura_anterior_ausente",
)
FONTES_SILVER = ("unidades_consumidoras", "leituras", "medidores", "geracao_distribuida", "inspecoes")

# Pesos de triagem NÃO CALIBRADOS; score não representa probabilidade.
PESO_QUEDA_PCT_MAX = 55.0
PESO_QUEDA_KWH_MAX = 30.0
PESO_ESTIMADAS = 10.0
PENALIDADE_GD_APOS_BASE = 35.0
PENALIDADE_TROCA = 30.0
PENALIDADE_INSPECAO = 25.0

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.dropdown("modo_exploracao", "false", ["false", "true"])
# Publicação explícita: false permite revisar; true publica após QA.
# Não usar este widget no notebook de exploração (que força publicação false).
dbutils.widgets.dropdown("publicar_gold", "true", ["false", "true"])
dbutils.widgets.dropdown("homologacao_regras", "true", ["false", "true"])
dbutils.widgets.text("versoes_silver_fixas_json", "")
# Valores efetivos, nunca inferidos dos defaults ou dos metadados do export.
PARAMETROS_NOMES = ("catalog", "modo_exploracao", "publicar_gold",
                    "homologacao_regras", "versoes_silver_fixas_json")

def ler_parametros_efetivos():
    return {nome: dbutils.widgets.get(nome) for nome in PARAMETROS_NOMES}

def decidir_publicacao(parametros):
    for nome in ("modo_exploracao", "publicar_gold", "homologacao_regras"):
        if parametros[nome] not in ("true", "false"):
            raise ValueError(f"{nome}: esperado exatamente true ou false")
    exploracao = parametros["modo_exploracao"] == "true"
    solicitada = parametros["publicar_gold"] == "true"
    homologada = parametros["homologacao_regras"] == "true"
    # Exploração tem precedência absoluta, inclusive diante de pedido de escrita.
    if exploracao:
        return False, "EXPLORACAO: escrita bloqueada por modo_exploracao=true"
    if solicitada and not homologada:
        raise RuntimeError("Publicação bloqueada: requer homologacao_regras=true")
    return solicitada, ("PUBLICACAO" if solicitada else "VALIDACAO: publicar_gold=false")

PARAMETROS_EFETIVOS = ler_parametros_efetivos()
print("Parâmetros efetivos:", PARAMETROS_EFETIVOS)
PUBLICAR_GOLD, MODO_EXECUCAO = decidir_publicacao(PARAMETROS_EFETIVOS)
MODO_EXPLORACAO = PARAMETROS_EFETIVOS["modo_exploracao"] == "true"
HOMOLOGACAO_REGRAS = PARAMETROS_EFETIVOS["homologacao_regras"] == "true"
CATALOGO = PARAMETROS_EFETIVOS["catalog"].strip()
PUBLICACAO_EFETIVADA = False
QA_EXECUCAO = None
AUTOTESTES_EXECUCAO = None
PUBLICACAO_TENTADA = False
print(f"Modo efetivo={MODO_EXECUCAO}; publicação autorizada={PUBLICAR_GOLD}")
print("A origem direta/%run/Job não é inferida dos widgets; conferir o histórico da execução no Databricks.")
import re
if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", CATALOGO):
    raise ValueError("Catálogo inválido: use identificador SQL simples sem caracteres especiais")
SILVER = f"{CATALOGO}.silver"
GOLD = f"{CATALOGO}.gold"

def versoes_fontes_silver():
    """Captura versões Delta de origem para auditoria e controle de alteração durante a carga."""
    versoes = {}
    for fonte in FONTES_SILVER:
        nome_tabela = f"{SILVER}.tb_{fonte}"
        try:
            linha = spark.sql(f"DESCRIBE HISTORY {nome_tabela} LIMIT 1").select("version").first()
        except Exception as erro:
            raise RuntimeError(f"Impossível consultar versão Delta de {nome_tabela}") from erro
        if linha is None:
            raise RuntimeError(f"Tabela Delta sem histórico: {nome_tabela}")
        versoes[fonte] = int(linha["version"])
    return versoes

import json
VERSOES_FONTES_HEAD_INICIO = versoes_fontes_silver()
VERSOES_FONTES_INICIO = dict(VERSOES_FONTES_HEAD_INICIO)
VERSOES_FIXAS_TEXTO = dbutils.widgets.get("versoes_silver_fixas_json").strip()
if VERSOES_FIXAS_TEXTO:
    try:
        fixas = json.loads(VERSOES_FIXAS_TEXTO)
        if (not isinstance(fixas, dict) or set(fixas) != set(FONTES_SILVER) or
            any(type(v) is not int or v < 0 or v > VERSOES_FONTES_HEAD_INICIO[k] for k, v in fixas.items())):
            raise ValueError("JSON de versões deve ter as cinco fontes com inteiros válidos.")
        VERSOES_FONTES_INICIO = fixas
    except (ValueError, TypeError) as exc:
        raise ValueError("versoes_silver_fixas_json inválido; copie versoes_silver_json da V6 para comparação.") from exc
    if PUBLICAR_GOLD and VERSOES_FONTES_INICIO != VERSOES_FONTES_HEAD_INICIO:
        raise RuntimeError("Publicação bloqueada em snapshot histórico: modo permitido apenas para comparação sem gravação.")
VERSOES_FONTES_JSON = json.dumps(VERSOES_FONTES_INICIO, sort_keys=True, ensure_ascii=False)
print("Silver HEAD capturada:", VERSOES_FONTES_HEAD_INICIO)
if VERSOES_FIXAS_TEXTO:
    print("MODO COMPARAÇÃO COM SNAPSHOT SILVER FIXO:", VERSOES_FONTES_INICIO)

def conferir_origens_inalteradas():
    atuais = versoes_fontes_silver()
    if atuais != VERSOES_FONTES_HEAD_INICIO:
        raise RuntimeError(
            f"Silver mudou durante o cálculo. HEAD antes={VERSOES_FONTES_HEAD_INICIO}; HEAD depois={atuais}. "
            "Não publicar; execute novamente a Gold com snapshot estável."
        )

def ler_silver(nome):
    # Lê cada fonte no MESMO snapshot registrado no início (Delta time travel).
    # Evita misturar versões durante o cálculo e permite auditoria reprodutível.
    versao = VERSOES_FONTES_INICIO[nome]
    return spark.sql(f"SELECT * FROM {SILVER}.tb_{nome} VERSION AS OF {versao}")

uc = ler_silver("unidades_consumidoras")
leituras = ler_silver("leituras")
medidores = ler_silver("medidores")
gd = ler_silver("geracao_distribuida")
inspecoes = ler_silver("inspecoes")

for nome, df in [("cadastro", uc), ("leituras", leituras),
                 ("medidores", medidores), ("GD", gd), ("inspecoes", inspecoes)]:
    print(f"{nome}: {', '.join(df.columns)}")
print(f"Regra={VERSAO_REGRA} | execução={ID_EXECUCAO} | publicar={PUBLICAR_GOLD}")
print(f"Versões das fontes Silver: {VERSOES_FONTES_JSON}")
print("Entradas: exclusivamente Silver. Gold Base não é utilizada.")


## 3. Contratos de entrada e integridade da Silver

O notebook verifica a existência das colunas necessárias e a unicidade do cadastro. O consumo é convertido para análise sem substituir os valores originais da Silver.

- Leituras com `id_uc`, competência ou consumo ausentes, consumo negativo, `flag_consumo_negativo` ou flag explícita de invalidade **não participam da comparação**. Se a flag de negativo marcar consumo não negativo, a execução falha para exigir validação da semântica com Théo.
- Para `id_uc + competência`, **duplicatas válidas geram erro** e devem ser corrigidas na Silver. Não usar `avg()` para esconder múltiplos registros.
- **A1:** `flag_consumo_inconsistente` é tratada como **arredondamento tolerado de até 1 kWh** (achado exploratório), rastreado em `tem_arredondamento_pareado` sem diminuir confiança nem bloquear candidatas. `flag_consumo_negativo` sinaliza valores a excluir do cálculo (reunião 21/10). `flag_leitura_anterior_ausente` permanece fator de revisão independente. Confirmar com Théo caso o contrato da Silver mude.
- A sinalização de outlier usa `flag_outlier` da Silver, se presente. Caso contrário, calcula um limite por classe (p99,9) com dados anteriores ou iguais ao corte, sem enxergar meses futuros no backtest. O método é registrado em `metodo_outlier`.
- A detecção de leitura estimada combina `flag_leitura_estimada` (nome real encontrado na Silver), alternativas legadas e `tipo_leitura`.
- **A5:** `flag_primeira_leitura_pos_troca` é verificada nos meses atuais **e nos respectivos meses da base**; retiradas são avaliadas desde o início da janela-base, sem descartar a leitura automaticamente.
- Eventos de GD, medidores e inspeções só são considerados até o **fechamento da competência de corte**. O catálogo Silver do cadastro é uma fotografia atual, não um histórico temporal completo.

As exclusões de leitura e os critérios de qualidade serão exibidos na seção de QA. Nenhum dado-fonte é regravado.


**Novo controle de rastreio:** cada tabela saída registra `executado_em_utc` e `versoes_silver_json`, permitindo identificar as versões Delta das cinco origens. Flags essenciais ausentes ou de tipo diferente de `boolean` agora são erros explícitos, e não são interpretadas silenciosamente como `False`.


In [0]:
def exigir(df, colunas, nome):
    faltantes = sorted(set(colunas) - set(df.columns))
    if faltantes:
        raise ValueError(f"{nome}: colunas da Silver ausentes: {faltantes}")

def flag(df, nome):
    if nome in df.columns:
        return F.coalesce(F.col(nome).cast("boolean"), F.lit(False))
    return F.lit(False)

exigir(uc, ["id_uc", "municipio", "bairro", "classe", "alimentador"], "cadastro")
exigir(leituras, ["id_uc", "mes_referencia", "consumo_kwh", "tipo_leitura",
                  "leitura_atual", "leitura_anterior", *FLAGS_OBRIGATORIAS], "leituras")
exigir(medidores, ["id_uc", "data_retirada"], "medidores")
exigir(gd, ["id_uc", "data_conexao"], "geracao_distribuida")
exigir(inspecoes, ["id_inspecao", "id_uc", "data", "resultado", "kwh_recuperado"], "inspecoes")

# Reconciliação A1 de flags e valores: a versão auditada tinha 1.682 leituras
# flagadas com divergência > 1 kWh, possivelmente relacionadas aos 1.712 negativos.
# Não pressupor essa relação: verificar o cruzamento contra a Silver atual.
delta_leitura = F.abs(
    F.col("consumo_kwh").cast("double") -
    (F.col("leitura_atual").cast("double") - F.col("leitura_anterior").cast("double"))
)
FLAG_INC = F.coalesce(F.col("flag_consumo_inconsistente"), F.lit(False))
FLAG_NEG = F.coalesce(F.col("flag_consumo_negativo"), F.lit(False))
LIMITE_ARREDONDAMENTO_KWH = 1.0
EPS_KWH = 1e-6
leitura_auditada = (leituras
    .withColumn("__dif_leitura", delta_leitura)
    .withColumn("__flag_inc", FLAG_INC)
    .withColumn("__flag_neg", FLAG_NEG)
    .withColumn("__negativo", F.col("consumo_kwh") < 0)
    .withColumn("__divergencia_material", F.col("__flag_inc") &
                (F.col("__dif_leitura") > LIMITE_ARREDONDAMENTO_KWH + EPS_KWH))
    .withColumn("__arredondamento_tolerado", F.col("__flag_inc") &
                (F.col("__dif_leitura") <= LIMITE_ARREDONDAMENTO_KWH + EPS_KWH))
)
resumo_flags_a1 = leitura_auditada.agg(
    F.count("*").alias("total_leituras"),
    F.sum(F.col("__flag_inc").cast("long")).alias("flag_inconsistente"),
    F.sum(F.col("__flag_neg").cast("long")).alias("flag_negativa"),
    F.sum(F.col("__negativo").cast("long")).alias("valor_negativo"),
    F.sum(F.col("__arredondamento_tolerado").cast("long")).alias("arredondamento_ate_1kwh"),
    F.sum(F.col("__divergencia_material").cast("long")).alias("divergencia_maior_1kwh"),
    F.sum((F.col("__divergencia_material") & (F.col("__negativo") | F.col("__flag_neg"))).cast("long")).alias("material_com_negativo"),
    F.sum((F.col("__divergencia_material") & ~F.col("__negativo") & ~F.col("__flag_neg")).cast("long")).alias("material_sem_negativo"),
    F.sum((F.col("__flag_inc") & F.col("__dif_leitura").isNull()).cast("long")).alias("flag_sem_comparacao")
).first().asDict()
print("Auditoria A1 da Silver (agregada):", resumo_flags_a1)
# Bloquear interpretações equivocadas. Sem resolução na Silver, não classificar como arredondamento.
if int(resumo_flags_a1["material_sem_negativo"] or 0):
    raise RuntimeError("A1: há consumo não negativo com diferença material (>1 kWh) e flag_cons_inconsistente; "
                       "não tratar como arredondamento. Validar flag com Théo e regras Silver.")
if int(resumo_flags_a1["flag_sem_comparacao"] or 0):
    raise RuntimeError("A1: há flag_cons_inconsistente sem leitura atual/anterior comparável; alinhar semântica com Théo.")
if int(resumo_flags_a1["flag_negativa"] or 0) != int(resumo_flags_a1["valor_negativo"] or 0):
    raise RuntimeError("Flag de consumo negativo não coincide em contagem com os valores negativos; alinhar com Théo.")

# Contratos físicos mínimos. A Silver já deveria entregar flags booleanas e números tipados.
from pyspark.sql.types import BooleanType
for nome_flag in FLAGS_OBRIGATORIAS:
    if not isinstance(leituras.schema[nome_flag].dataType, BooleanType):
        raise TypeError(f"Silver.leituras.{nome_flag} deve ser boolean, não {leituras.schema[nome_flag].dataType}")
if "flag_outlier" in leituras.columns and not isinstance(leituras.schema["flag_outlier"].dataType, BooleanType):
    raise TypeError("Silver.leituras.flag_outlier deve ser boolean")

assert uc.where(F.col("id_uc").isNull()).limit(1).count() == 0, "UC nula no cadastro"
assert uc.groupBy("id_uc").count().where(F.col("count") > 1).limit(1).count() == 0, "UC duplicada no cadastro"

flags_disponiveis = [c for c in ["flag_estimada", "flag_leitura_estimada", "flag_outlier",
                                  "flag_leitura_invalida", "flag_consumo_invalido",
                                  "flag_primeira_leitura_pos_troca", "flag_consumo_inconsistente",
                                  "flag_consumo_negativo", "flag_leitura_anterior_ausente"]
                      if c in leituras.columns]
leitura_mensal = (leituras
    .select("id_uc", "mes_referencia", "consumo_kwh", "leitura_atual", "leitura_anterior",
            "tipo_leitura", *flags_disponiveis)
    .withColumn("mes", F.to_date(F.concat(
        F.substring(F.col("mes_referencia").cast("string"), 1, 7), F.lit("-01"))))
    .withColumn("consumo", F.col("consumo_kwh").cast("double"))
    .withColumn("estimada", flag(leituras, "flag_estimada") |
                flag(leituras, "flag_leitura_estimada") |
                (F.upper(F.trim(F.col("tipo_leitura").cast("string"))) == "ESTIMADA"))
    .withColumn("estimada", F.coalesce(F.col("estimada"), F.lit(False)))
    .withColumn("outlier_silver", flag(leituras, "flag_outlier"))
    .withColumn("pos_troca", flag(leituras, "flag_primeira_leitura_pos_troca"))
    .withColumn("invalida", flag(leituras, "flag_leitura_invalida") |
                flag(leituras, "flag_consumo_invalido"))
    # A1: só tratar como arredondamento a diferença observada de ATÉ 1 kWh.
    # Os casos com divergência material não são silenciosamente absorvidos.
    .withColumn("arredondamento_tolerado", flag(leituras, "flag_consumo_inconsistente") &
                (F.abs(F.col("consumo_kwh").cast("double") -
                       (F.col("leitura_atual").cast("double") - F.col("leitura_anterior").cast("double")))
                 <= LIMITE_ARREDONDAMENTO_KWH + EPS_KWH))
    # Reunião 21/10: sinalização negativa remanescente impede uso nos cálculos Gold.
    .withColumn("consumo_negativo_sinalizado", flag(leituras, "flag_consumo_negativo"))
    .withColumn("sem_leitura_anterior", flag(leituras, "flag_leitura_anterior_ausente"))
)

condicao_invalida = (
    F.col("id_uc").isNull() | F.col("mes").isNull() |
    F.col("consumo").isNull() | (F.col("consumo") < 0) |
    F.col("consumo_negativo_sinalizado") | F.col("invalida")
)
qtd_excluidas = leitura_mensal.where(condicao_invalida).count()
mensal = (leitura_mensal.where(~condicao_invalida)
    .select("id_uc", "mes", "consumo", "estimada", "outlier_silver", "pos_troca",
            "arredondamento_tolerado", "sem_leitura_anterior")
    .join(uc.select("id_uc", "classe"), "id_uc", "left"))

# A Gold não oculta UCs órfãs nem perda da classe durante as junções.
assert mensal.where(F.col("classe").isNull()).limit(1).count() == 0, (
    "Leituras sem classe no cadastro Silver; revisar integridade referencial na Silver")

METODO_OUTLIER = ("FLAG_SILVER" if "flag_outlier" in leituras.columns
                  else "P999_CLASSE_HISTORICO_ATE_CORTE")
if METODO_OUTLIER != "FLAG_SILVER":
    print("AVISO: flag_outlier não existe na Silver. Será usado p99,9 por classe "+
          "até cada corte, sem escrever na Silver.")
print(f"Detecção de outliers: {METODO_OUTLIER}")

repeticoes = mensal.groupBy("id_uc", "mes").count().where(F.col("count") > 1)
if repeticoes.limit(1).count():
    display(repeticoes.orderBy(F.desc("count")).limit(10))
    raise ValueError("Leituras Silver com UC+mês duplicados: corrigir na Silver, não aplicar avg() na Gold")

referencia = mensal.agg(F.max("mes").alias("mes_max")).first()["mes_max"]
assert referencia is not None, "Nenhuma competência válida na Silver de leituras"
assert uc.limit(1).count() > 0, "Silver de cadastro está vazia"
qtd_uc_competencia = mensal.where(F.col("mes") == F.lit(referencia)).select("id_uc").distinct().count()
cobertura_competencia = qtd_uc_competencia / uc.count()
if cobertura_competencia < MIN_COBERTURA_ULTIMO_MES:
    raise ValueError(
        f"Competência {referencia} parece incompleta: {qtd_uc_competencia} UCs com leitura "
        f"({cobertura_competencia:.1%}), abaixo de {MIN_COBERTURA_ULTIMO_MES:.0%}. "
        "Revisar fechamento mensal na Silver antes da publicação."
    )
print(f"Cobertura da competência de referência: {cobertura_competencia:.1%} ({qtd_uc_competencia} UCs)")
print(f"Referência: {referencia} | leituras inválidas fora da análise: {qtd_excluidas}")
# Validar a semântica acordada com Théo: flags remanescentes em valores não negativos
# também ficam fora até confirmação; mostrar contagem agregada sem divulgar IDs.
negativos_divergentes = leitura_mensal.where(
    F.col("consumo_negativo_sinalizado") & (F.col("consumo") >= 0)).count()
if negativos_divergentes:
    print(f"ATENÇÃO: {negativos_divergentes} linhas com flag negativa e consumo >= 0; "
          "confirmar semântica da flag com Théo antes de publicar.")
    # Bloqueio protege contra descartar consumos corrigidos que permaneçam flagados.
    raise ValueError("Semântica ambígua da flag_consumo_negativo: alinhar com Silver de medição")

# Datas já tipadas na Silver. Não levar dados futuros para o score retrospectivo.
gd_datas = gd.select("id_uc", F.col("data_conexao").cast("date").alias("data_conexao"))
medidores_datas = medidores.select("id_uc", F.col("data_retirada").cast("date").alias("data_retirada"))
insp_base = inspecoes.select(
    "id_inspecao", "id_uc", F.col("data").cast("date").alias("data_inspecao"),
    F.upper(F.trim(F.col("resultado"))).alias("resultado"),
    F.col("kwh_recuperado").cast("double").alias("kwh_recuperado"))


## 4. Metodologia da fila de perdas

### 4.1. Consumo e comparação sazonal

Para cada UC, parear cada mês dos **3 meses de referência** ao mesmo mês do ano anterior, considerando apenas pares em que ambos os meses possuem consumo válido. A diferença mensal média positiva é:

\[\Delta kWh_{mês} = \max(0,\overline{kWh}_{ano\ anterior} - \overline{kWh}_{atual})\]

A queda relativa só é calculada quando a média anterior é positiva. O potencial bruto é a diferença mensal multiplicada pela **quantidade de meses efetivamente pareados**, não automaticamente por três.

### 4.2. Score e evidências

| Critério | Tratamento |
| --- | --- |
| Queda percentual de consumo | Até 55 pontos |
| Queda absoluta média | Até 30 pontos |
| Pelo menos dois **pares comparáveis** com leitura estimada | +10 pontos |
| GD conectada após o início da janela-base | −35 pontos e retenção para revisão |
| Troca de medidor desde início da janela-base ou leitura pós-troca (base/atual) | −30 pontos e retenção para revisão |
| Inspeção nos últimos 12 meses antes do corte | −25 pontos e status de revisão próprio |

Os pontos de queda dependem da existência de pares válidos. As leituras estimadas servem como **sinal de auditoria**, não como prova de perda. A ausência de leitura anterior é exibida como fator de revisão, não convertida em zero; arredondamentos tolerados **não** são fatores de retenção. As evidências são classificadas como `ALTA`, `MEDIA`, `BAIXA` ou `INDETERMINADA` conforme a quantidade de pares medidos, estimados e com flags de qualidade.

### 4.3. Decisão de triagem

O campo `status_triagem` separa `ELEGIVEL`, `HISTORICO_INSUFICIENTE`, `SEM_EVIDENCIA_MINIMA`, `REVISAR_EVIDENCIA_ESTIMADA`, `REVISAR_FATORES_ALTERNATIVOS`, `REVISAR_INSPECAO_RECENTE` e `BAIXA_PRIORIDADE`. Apenas `ELEGIVEL` entra na fila operacional; os demais permanecem na auditoria, com `motivos`, `fatores_atenuantes` e métricas de suporte.

> **Limitações:** queda de consumo pode decorrer de sazonalidade residual, ocupação, mudança de atividade, eficiência energética, medidores ou GD. Queda absoluta isolada pode favorecer unidades de grande porte; por isso o motivo de queda absoluta agora também exige queda percentual mínima de 20%, proposta sujeita a calibração. Os pesos são hipóteses de triagem e precisam de validação futura.


### 4.4. Organização do processamento (regras revisadas na V10)

A função `calcular_gold(mes_ref)` apenas **orquestra** as etapas do processamento; cada etapa pode ser lida, testada e mantida isoladamente:

| Função | Responsabilidade |
| --- | --- |
| `preparar_pares_consumo` | Parear cada competência atual à mesma competência do ano anterior. |
| `resumir_consumo_sazonal` | Agregar médias, quantidade de pares e indicadores de qualidade. |
| `reunir_contexto_uc` | Acrescentar cadastro, conexão de GD, trocas de medidor e inspeções anteriores ao corte. |
| `avaliar_evidencias_consumo` | Determinar suficiência do histórico, confiança e quedas de consumo. |
| `calcular_score` | Manter pesos numéricos, ampliar a janela da penalidade de inspeção e da troca. |
| `definir_elegibilidade` | Produzir motivos, fatores atenuantes, status e estimativas em kWh. |
| `selecionar_colunas_gold` | Definir o esquema da saída e textos JSON auxiliares para exportação. |

**Controle de mudança V10:** as fontes Silver, fórmula do proxy kWh, estrutura de publicação e pesos numéricos permanecem. Mudam semântica de arredondamento (A1), histórico/status de inspeções (A2), limiar para motivo absoluto (A3) e abrangência de trocas (A5). A V4/V6 publicada é tratada como baseline (A4); **não é possível antecipar o novo número de elegíveis sem executar a V10**.


In [0]:
def preparar_consumo_ate_corte(ref):
    """Aplica outlier por classe sem olhar competências futuras do corte."""
    historico = mensal.where(F.col("mes") <= ref)
    if METODO_OUTLIER == "FLAG_SILVER":
        return historico.withColumn("outlier", F.col("outlier_silver"))

    # Indicador ANALÍTICO desta Gold (não limpeza Silver). Não usa a tabela Gold Base.
    limites = (historico.groupBy("classe")
        .agg(F.expr(f"percentile_approx(consumo, {PERCENTIL_OUTLIER_CLASSE}, "
                    f"{PRECISAO_PERCENTIL_OUTLIER})").alias("limite_classe")))
    return (historico.join(limites, "classe", "left")
        .withColumn("outlier", F.coalesce(F.col("consumo") > F.col("limite_classe"),
                                           F.lit(False)))
        .drop("limite_classe"))


def preparar_pares_consumo(ref, base_inicio):
    """Pareia meses de consumo com os mesmos meses do ano anterior."""
    historico = preparar_consumo_ate_corte(ref)
    atual = historico.where((F.col("mes") >= F.add_months(ref, -(JANELA_MESES - 1))) &
                           (F.col("mes") <= ref))
    ano_passado = (historico
        .where((F.col("mes") >= base_inicio) & (F.col("mes") <= F.add_months(ref, -12)))
        .select("id_uc", F.add_months(F.col("mes"), 12).alias("mes"),
                F.col("consumo").alias("consumo_base"),
                F.col("estimada").alias("estimada_base"),
                F.col("outlier").alias("outlier_base"),
                F.col("arredondamento_tolerado").alias("arredondamento_tolerado_base"),
                F.col("pos_troca").alias("pos_troca_base"),
                F.col("sem_leitura_anterior").alias("sem_leitura_anterior_base")))

    pares = (atual.join(ano_passado, ["id_uc", "mes"], "left")
        .withColumn("par_valido", F.col("consumo_base").isNotNull())
        .withColumn("par_estimado", F.col("par_valido") &
                    (F.col("estimada") | F.coalesce(F.col("estimada_base"), F.lit(False))))
        .withColumn("par_observado", F.col("par_valido") & ~F.col("par_estimado"))
        .withColumn("par_arredondamento_tolerado", F.col("par_valido") &
                    (F.col("arredondamento_tolerado") |
                     F.coalesce(F.col("arredondamento_tolerado_base"), F.lit(False))))
        .withColumn("par_pos_troca_base", F.col("par_valido") &
                    F.coalesce(F.col("pos_troca_base"), F.lit(False)))
        .withColumn("par_sem_leitura_anterior", F.col("par_valido") &
                    (F.col("sem_leitura_anterior") |
                     F.coalesce(F.col("sem_leitura_anterior_base"), F.lit(False)))))

    return pares


def resumir_consumo_sazonal(pares):
    """Consolida quantidades de pares, medias e flags por UC."""
    resumo = pares.groupBy("id_uc").agg(
        F.sum(F.col("par_valido").cast("int")).alias("meses_comparaveis"),
        F.count("*").alias("meses_atuais"),
        F.avg(F.when(F.col("par_valido"), F.col("consumo"))).alias("media_atual"),
        F.avg("consumo_base").alias("media_anterior"),
        F.sum(F.col("estimada").cast("int")).alias("meses_estimados"),
        F.sum(F.col("par_estimado").cast("int")).alias("pares_estimados"),
        F.sum(F.col("par_observado").cast("int")).alias("pares_observados"),
        F.max(F.col("par_arredondamento_tolerado").cast("int")).alias("tem_arredondamento_pareado"),
        F.max(F.col("par_pos_troca_base").cast("int")).alias("troca_sinalizada_base"),
        F.max(F.col("par_sem_leitura_anterior").cast("int")).alias("tem_sem_leitura_anterior_pareada"),
        F.max(F.when(F.col("par_valido"),
                     (F.col("outlier") | F.coalesce(F.col("outlier_base"), F.lit(False))))
              .otherwise(F.lit(False)).cast("int")).alias("tem_outlier_pareado"),
        F.max(F.col("pos_troca").cast("int")).alias("troca_sinalizada_silver"))

    return resumo


def reunir_contexto_uc(resumo, mes_ref, ref, fim, base_inicio):
    """Reune cadastro, GD, trocas e inspecoes ate a data de corte."""
    # Na data de corte, considerar apenas eventos conhecidos até aquele momento.
    gd_uc = (gd_datas.where(F.col("data_conexao").isNotNull() & (F.col("data_conexao") <= fim))
        .groupBy("id_uc").agg(F.min("data_conexao").alias("primeira_conexao_gd")))
    gd_data_desconhecida = (gd_datas.where(F.col("data_conexao").isNull() &
                                     F.lit(mes_ref == referencia))
        .select("id_uc").distinct().withColumn("gd_data_desconhecida", F.lit(True)))
    trocas_uc = (medidores_datas
        .where(F.col("data_retirada").between(base_inicio, fim))
        .groupBy("id_uc").agg(F.count("*").alias("trocas_recentes")))
    from pyspark.sql import Window
    w_ultima_inspecao = Window.partitionBy("id_uc").orderBy(
        F.desc("data_inspecao"), F.desc("id_inspecao"))
    insp_passadas = (insp_base
        .where(F.col("data_inspecao").isNotNull() & (F.col("data_inspecao") <= fim))
        .withColumn("ordem_ultima", F.row_number().over(w_ultima_inspecao))
        .where(F.col("ordem_ultima") == 1)
        .select("id_uc", F.col("data_inspecao").alias("ultima_inspecao"),
                F.col("resultado").alias("resultado_ultima_inspecao")))

    universo = (uc.select("id_uc", "municipio", "bairro", "classe", "alimentador")
        .join(resumo, "id_uc", "left")
        .join(gd_uc, "id_uc", "left")
        .join(gd_data_desconhecida, "id_uc", "left")
        .join(trocas_uc, "id_uc", "left")
        .join(insp_passadas, "id_uc", "left"))

    return universo


def avaliar_evidencias_consumo(universo, base_inicio, fim):
    """Mede confianca, fatores alternativos e quedas sazonais."""
    universo = (universo
        .fillna(0, subset=["meses_comparaveis", "meses_atuais", "meses_estimados",
                           "pares_estimados", "pares_observados"])
        .withColumn("historico_suficiente", F.col("meses_comparaveis") >= MIN_PARES)
        .withColumn("tem_outlier_pareado", F.coalesce(F.col("tem_outlier_pareado") > 0, F.lit(False)))
        .withColumn("tem_arredondamento_pareado", F.coalesce(F.col("tem_arredondamento_pareado") > 0, F.lit(False)))
        .withColumn("tem_sem_leitura_anterior_pareada", F.coalesce(
            F.col("tem_sem_leitura_anterior_pareada") > 0, F.lit(False)))
        .withColumn("proporcao_pares_estimados", F.when(F.col("meses_comparaveis") > 0,
            F.col("pares_estimados") / F.col("meses_comparaveis")).otherwise(F.lit(None).cast("double")))
        .withColumn("confianca_evidencia", F.when(~F.col("historico_suficiente"), "INDETERMINADA")
            .when(F.col("pares_observados") < MIN_PARES_OBSERVADOS, "BAIXA")
            .when((F.col("proporcao_pares_estimados") >= LIMIAR_PROP_PARES_ESTIMADOS) |
                  F.col("tem_outlier_pareado") |
                  F.col("tem_sem_leitura_anterior_pareada"), "MEDIA")
            .otherwise("ALTA"))
        .withColumn("gd_data_desconhecida", F.coalesce(F.col("gd_data_desconhecida"), F.lit(False)))
        .withColumn("possui_gd", F.col("primeira_conexao_gd").isNotNull() | F.col("gd_data_desconhecida"))
        .withColumn("gd_apos_base", F.coalesce(F.col("primeira_conexao_gd").between(base_inicio, fim), F.lit(False)))
        .withColumn("houve_troca_na_janela", F.coalesce(F.col("trocas_recentes") > 0, F.lit(False)) |
                    F.coalesce(F.col("troca_sinalizada_silver") > 0, F.lit(False)) |
                    F.coalesce(F.col("troca_sinalizada_base") > 0, F.lit(False)))
        .withColumn("ja_inspecionada", F.col("ultima_inspecao").isNotNull())
        .withColumn("inspecao_recente", F.coalesce(F.col("ultima_inspecao") >=
                    F.add_months(fim, -MESES_INSPECAO_RECENTE), F.lit(False))))

    universo = (universo
        .withColumn("queda_kwh_mensal", F.when(F.col("historico_suficiente") &
             (F.col("media_anterior") > 0),
             F.greatest(F.lit(0.0), F.col("media_anterior") - F.col("media_atual")))
             .otherwise(F.lit(0.0)))
        .withColumn("queda_pct", F.when(F.col("historico_suficiente") &
             (F.col("media_anterior") > 0),
             F.greatest(F.lit(0.0), (F.col("media_anterior") - F.col("media_atual")) /
                        F.col("media_anterior"))).otherwise(F.lit(0.0))))

    return universo


def calcular_score(universo):
    """Aplica pesos e penalidades existentes, sem recalibracao."""
    universo = (universo
        .withColumn("p_queda_pct", F.when(F.col("historico_suficiente"),
            F.least(F.lit(PESO_QUEDA_PCT_MAX), F.col("queda_pct") * 100.0))
            .otherwise(F.lit(0.0)))
        .withColumn("p_queda_kwh", F.when(
            F.col("historico_suficiente") &
            (F.col("queda_pct") >= LIMIAR_PCT_PARA_QUEDA_ABSOLUTA) &
            (F.col("queda_kwh_mensal") >= LIMIAR_ABS_ELEVADO_KWH_MES),
            F.least(F.lit(PESO_QUEDA_KWH_MAX),
                    F.col("queda_kwh_mensal") * (PESO_QUEDA_KWH_MAX / 300.0)))
            .otherwise(F.lit(0.0)))
        .withColumn("p_estimada", F.when(F.col("historico_suficiente") &
            (F.col("pares_estimados") >= 2), F.lit(PESO_ESTIMADAS)).otherwise(F.lit(0.0)))
        .withColumn("score_bruto", F.col("p_queda_pct") + F.col("p_queda_kwh") + F.col("p_estimada"))
        .withColumn("penalidades",
            F.when(F.col("gd_apos_base"), F.lit(PENALIDADE_GD_APOS_BASE)).otherwise(F.lit(0.0)) +
            F.when(F.col("houve_troca_na_janela"), F.lit(PENALIDADE_TROCA)).otherwise(F.lit(0.0)) +
            F.when(F.col("inspecao_recente"), F.lit(PENALIDADE_INSPECAO)).otherwise(F.lit(0.0)))
        .withColumn("score", F.round(F.greatest(F.lit(0.0),
            F.least(F.lit(100.0), F.col("score_bruto") - F.col("penalidades"))), 2)))

    return universo


def definir_elegibilidade(universo, ref):
    """Produz motivos, atenuantes, status e proxies de kWh."""
    universo = (universo
        .withColumn("motivos", F.array(
            F.when((F.col("queda_pct") >= LIMIAR_QUEDA_PCT) &
                   (F.col("queda_kwh_mensal") >= LIMIAR_QUEDA_KWH_MES),
                   F.lit("QUEDA_SAZONAL_RELEVANTE")),
            F.when((F.col("queda_kwh_mensal") >= LIMIAR_ABS_ELEVADO_KWH_MES) &
                   (F.col("queda_pct") >= LIMIAR_PCT_PARA_QUEDA_ABSOLUTA),
                   F.lit("QUEDA_ABSOLUTA_RELEVANTE")),
            F.when(F.col("historico_suficiente") & (F.col("pares_estimados") >= 2),
                   F.lit("LEITURAS_ESTIMADAS_RECORRENTES"))))
        .withColumn("motivos", F.expr("filter(motivos, x -> x is not null)"))
        .withColumn("fatores_atenuantes", F.array(
            F.when(F.col("gd_apos_base"), F.lit("GD_CONECTADA_APOS_BASE")),
            F.when(F.col("gd_data_desconhecida"), F.lit("GD_SEM_DATA_CONEXAO")),
            F.when(F.col("houve_troca_na_janela"), F.lit("TROCA_MEDIDOR_NA_JANELA_BASE_ATUAL")),
            F.when(F.col("inspecao_recente"), F.lit("INSPECAO_NOS_ULTIMOS_12_MESES")),
            F.when(F.col("ja_inspecionada") & ~F.col("inspecao_recente"),
                   F.lit("INSPECAO_ANTERIOR_FORA_JANELA")),
            F.when(F.col("tem_outlier_pareado"), F.lit("OUTLIER_NO_PAREAMENTO")),
            F.when(F.col("tem_sem_leitura_anterior_pareada"), F.lit("LEITURA_ANTERIOR_AUSENTE")),
            F.when(F.col("pares_observados") < MIN_PARES_OBSERVADOS,
                   F.lit("AUSENCIA_PARES_MEDIDOS"))))
        .withColumn("fatores_atenuantes", F.expr("filter(fatores_atenuantes, x -> x is not null)")))

    universo = (universo
        .withColumn("status_triagem",
            F.when(~F.col("historico_suficiente"), F.lit("HISTORICO_INSUFICIENTE"))
             .when(F.size("motivos") == 0, F.lit("SEM_EVIDENCIA_MINIMA"))
             .when(F.col("pares_observados") < MIN_PARES_OBSERVADOS,
                   F.lit("REVISAR_EVIDENCIA_ESTIMADA"))
             .when(F.col("gd_apos_base") | F.col("gd_data_desconhecida") |
                   F.col("houve_troca_na_janela") | F.col("tem_outlier_pareado") |
                   F.col("tem_sem_leitura_anterior_pareada"),
                   F.lit("REVISAR_FATORES_ALTERNATIVOS"))
             .when(F.col("inspecao_recente"), F.lit("REVISAR_INSPECAO_RECENTE"))
             .when(F.col("score") < MIN_SCORE_ELEGIVEL, F.lit("BAIXA_PRIORIDADE"))
             .otherwise(F.lit("ELEGIVEL")))
        # Faixa operacional NÃO altera o total elegível: distingue ação de campo
        # de revisão prévia de medição; estimadas não são prova independente de perdas.
        .withColumn("queda_qualificada", F.array_contains(F.col("motivos"), "QUEDA_SAZONAL_RELEVANTE") |
                    F.array_contains(F.col("motivos"), "QUEDA_ABSOLUTA_RELEVANTE"))
        .withColumn("acao_recomendada", F.when(F.col("status_triagem") != "ELEGIVEL", F.lit("NAO_APLICAVEL"))
                    .when(F.col("queda_qualificada"), F.lit("INSPECAO_PRIORITARIA"))
                    .otherwise(F.lit("VALIDACAO_DOCUMENTAL")))
        .withColumn("prioridade_operacional", F.when(F.col("acao_recomendada") == "INSPECAO_PRIORITARIA", F.lit(1))
                    .when(F.col("acao_recomendada") == "VALIDACAO_DOCUMENTAL", F.lit(2))
                    .otherwise(F.lit(9)))
        .withColumn("kwh_potencial_bruto", F.round(
            F.col("queda_kwh_mensal") * F.col("meses_comparaveis"), 2))
        .withColumn("kwh_estimado", F.when(F.col("status_triagem") == "ELEGIVEL",
            F.col("kwh_potencial_bruto")).otherwise(F.lit(0.0)))
        .withColumn("mes_referencia", ref)
        .withColumn("versao_regra", F.lit(VERSAO_REGRA))
        .withColumn("id_execucao", F.lit(ID_EXECUCAO))
        .withColumn("executado_em_utc", F.lit(INSTANTE_EXECUCAO_UTC).cast("timestamp"))
        .withColumn("versoes_silver_json", F.lit(VERSOES_FONTES_JSON))
        .withColumn("metodo_estimativa", F.lit("DIFERENCA_MEDIA_SAZONAL_X_MESES_PAREADOS"))
        .withColumn("metodo_outlier", F.lit(METODO_OUTLIER))
        .withColumn("motivos_json", F.to_json(F.col("motivos")))
        .withColumn("fatores_atenuantes_json", F.to_json(F.col("fatores_atenuantes"))))

    return universo


def selecionar_colunas_gold(universo):
    """Preserva o contrato e explicita duas ações distintas na fila operacional."""
    colunas = ["id_uc", "municipio", "bairro", "classe", "alimentador", "score",
               "score_bruto", "p_queda_pct", "p_queda_kwh", "p_estimada",
               "queda_qualificada", "acao_recomendada", "prioridade_operacional",
               "penalidades", "motivos", "fatores_atenuantes",
               "kwh_estimado", "kwh_potencial_bruto", "queda_pct", "queda_kwh_mensal",
               "media_anterior", "media_atual", "meses_atuais", "meses_comparaveis",
               "meses_estimados", "pares_estimados", "pares_observados",
               "proporcao_pares_estimados", "confianca_evidencia", "historico_suficiente",
               "possui_gd", "primeira_conexao_gd", "gd_apos_base", "gd_data_desconhecida",
               "houve_troca_na_janela", "tem_outlier_pareado", "tem_arredondamento_pareado",
               "tem_sem_leitura_anterior_pareada", "metodo_outlier",
               "motivos_json", "fatores_atenuantes_json",
               "ultima_inspecao", "resultado_ultima_inspecao", "ja_inspecionada",
               "inspecao_recente", "status_triagem", "mes_referencia",
               "versao_regra", "id_execucao", "executado_em_utc", "versoes_silver_json",
               "metodo_estimativa"]
    return (universo.select(*colunas)
        .withColumn("kwh_estimado", F.col("kwh_estimado").cast("decimal(14,2)"))
        .withColumn("kwh_potencial_bruto", F.col("kwh_potencial_bruto").cast("decimal(14,2)")))


def calcular_gold(mes_ref):
    """Orquestra a Gold final para uma competência de corte."""
    ref = F.lit(mes_ref).cast("date")
    fim = F.last_day(ref)
    base_inicio = F.add_months(ref, -(12 + JANELA_MESES - 1))

    pares = preparar_pares_consumo(ref, base_inicio)
    resumo = resumir_consumo_sazonal(pares)
    universo = reunir_contexto_uc(resumo, mes_ref, ref, fim, base_inicio)
    universo = avaliar_evidencias_consumo(universo, base_inicio, fim)
    universo = calcular_score(universo)
    universo = definir_elegibilidade(universo, ref)
    return selecionar_colunas_gold(universo)

universo_atual = calcular_gold(referencia)
# Ranking materializado somente entre candidatas elegíveis; desempate estável e documentado.
from pyspark.sql import Window
JANELA_RANKING = Window.partitionBy(F.lit(1)).orderBy(
    F.asc("prioridade_operacional"), F.desc("score"), F.desc("kwh_estimado"), F.asc("id_uc"))
fila = (universo_atual.where(F.col("status_triagem") == "ELEGIVEL")
    .withColumn("posicao_fila", F.row_number().over(JANELA_RANKING)))
print("Universo e fila calculados; ainda não publicados na Gold.")

## 5. Garantias de qualidade (QA bloqueante)

Antes de publicar, validar cardinalidade de 45.000 UCs conforme cadastro atual, unicidade, integridade, status, limites de score, consumo válido, alinhamento entre elegibilidade e fila, ordenação, arredondamento não bloqueante, inspeção recente e condição percentual do motivo absoluto.

Qualquer assert que falhar **interrompe antes da gravação**. Gráficos, recortes por classe e município, comparação entre versões e backtest ficam em `exploracao_gold_fila_perdas`.


In [0]:
total_ucs = uc.count()
assert universo_atual.count() == total_ucs, "Cardinalidade incorreta: joins replicaram ou descartaram UCs"
assert universo_atual.groupBy("id_uc").count().where(F.col("count") > 1).limit(1).count() == 0, "UC duplicada na Gold"
assert universo_atual.join(uc.select("id_uc"), "id_uc", "left_anti").limit(1).count() == 0, "UC órfã na Gold"
assert universo_atual.where(
    F.col("score").isNull() | (F.col("score") < 0) | (F.col("score") > 100) |
    F.col("kwh_estimado").isNull() | (F.col("kwh_estimado") < 0) |
    F.col("kwh_potencial_bruto").isNull() | (F.col("kwh_potencial_bruto") < 0) |
    F.col("status_triagem").isNull() | F.col("confianca_evidencia").isNull() |
    F.col("metodo_outlier").isNull() | F.col("motivos_json").isNull() |
    F.col("fatores_atenuantes_json").isNull() |
    (F.col("meses_comparaveis") > JANELA_MESES) |
    (F.col("pares_estimados") < 0) | (F.col("pares_observados") < 0) |
    ((F.col("kwh_estimado") > 0) & (F.col("status_triagem") != "ELEGIVEL")) |
    (F.col("pares_observados") + F.col("pares_estimados") != F.col("meses_comparaveis"))
).limit(1).count() == 0, "Score, evidência, contagens ou kWh inválidos"
assert fila.where((F.size("motivos") == 0) | (~F.col("historico_suficiente")) |
    (F.col("pares_observados") < MIN_PARES_OBSERVADOS) |
    (F.col("score") < MIN_SCORE_ELEGIVEL) | F.col("gd_apos_base") |
    F.col("gd_data_desconhecida") | F.col("houve_troca_na_janela") |
    F.col("inspecao_recente") | F.col("tem_outlier_pareado") |
    F.col("tem_sem_leitura_anterior_pareada")).limit(1).count() == 0, "Fila contém UC inelegível"

qtd_candidatos = fila.count()
print(f"UCs cadastro={total_ucs} | elegíveis={qtd_candidatos} | referência={referencia}")
print(f"Método de outliers: {METODO_OUTLIER} | consumo negativo fora das médias;")
print("arredondamento tolerado em campo informativo (nunca bloqueante).")
assert universo_atual.where(F.col("inspecao_recente") & ~F.col("ja_inspecionada")).limit(1).count() == 0
assert fila.where(F.col("inspecao_recente")).limit(1).count() == 0
assert universo_atual.where(F.col("tem_arredondamento_pareado").isNull()).limit(1).count() == 0
assert universo_atual.where(
    F.array_contains(F.col("motivos"), "QUEDA_ABSOLUTA_RELEVANTE") &
    (F.col("queda_pct") < LIMIAR_PCT_PARA_QUEDA_ABSOLUTA)
).limit(1).count() == 0
# Integridade cruzada da publicação e consistência do novo ranking.
assert fila.where(F.col("posicao_fila").isNull() | (F.col("posicao_fila") < 1)).limit(1).count() == 0
assert fila.groupBy("posicao_fila").count().where(F.col("count") > 1).limit(1).count() == 0
assert fila.where(F.col("posicao_fila") > qtd_candidatos).limit(1).count() == 0
assert fila.join(universo_atual.where(F.col("status_triagem") == "ELEGIVEL").select("id_uc"),
                 "id_uc", "left_anti").limit(1).count() == 0
assert fila.select("id_uc").distinct().count() == qtd_candidatos
assert universo_atual.where(
    (F.col("versoes_silver_json") != VERSOES_FONTES_JSON) |
    (F.col("executado_em_utc").isNull()) |
    (F.col("id_execucao") != ID_EXECUCAO)
).limit(1).count() == 0, "Rastreabilidade inválida na Gold"
print("QA cruzado: ranking sequencial, versão das fontes e id_execucao coerentes.")

# A3: a correção atinge o componente de score, não apenas o motivo textual.
assert universo_atual.where(
    (F.col("p_queda_kwh") > 0) &
    ((F.col("queda_pct") < LIMIAR_PCT_PARA_QUEDA_ABSOLUTA) |
     (F.col("queda_kwh_mensal") < LIMIAR_ABS_ELEVADO_KWH_MES))
).limit(1).count() == 0, "A3: pontos por queda absoluta sem limiar percentual/absoluto"
industria_top200 = fila.where((F.col("posicao_fila") <= TOP_N) & (F.col("classe") == "INDUSTRIAL"))
indicador_industria = industria_top200.agg(
    F.count("*").alias("industriais_top200"),
    F.round(F.avg(F.col("queda_pct") * 100), 2).alias("queda_pct_media"),
    F.sum((F.col("queda_pct") < LIMIAR_PCT_PARA_QUEDA_ABSOLUTA).cast("long")).alias("industriais_queda_baixa")
).first().asDict()
print("A3 top 200 (após recalibração de score):", indicador_industria)
print("NOTA: ainda exige avaliação de Rafael; o teste prova a regra do score, não a taxa de acerto.")

# A6 definitivo: as UCs de estimadas continuam disponíveis, com ação diferenciada.
assert universo_atual.where(F.col("acao_recomendada").isNull() |
    F.col("prioridade_operacional").isNull() |
    F.col("queda_qualificada").isNull()).limit(1).count() == 0
assert fila.where(
    (F.col("acao_recomendada") == "INSPECAO_PRIORITARIA") &
    (~F.col("queda_qualificada"))
).limit(1).count() == 0, "UC indicada para campo sem queda qualificada"
assert fila.where(
    (F.col("acao_recomendada") == "VALIDACAO_DOCUMENTAL") &
    (F.col("queda_qualificada") |
     ~F.array_contains(F.col("motivos"), "LEITURAS_ESTIMADAS_RECORRENTES"))
).limit(1).count() == 0, "Revisão documental sem justificativa de estimadas"
assert fila.where(~F.col("acao_recomendada").isin("INSPECAO_PRIORITARIA", "VALIDACAO_DOCUMENTAL")
).limit(1).count() == 0, "Fila sem ação operacional válida"
assert universo_atual.where((F.col("status_triagem") != "ELEGIVEL") &
    (F.col("acao_recomendada") != "NAO_APLICAVEL")
).limit(1).count() == 0, "Ação operacional atribuída a UC fora da fila"
resumo_operacional = {r["acao_recomendada"]: r["count"] for r in
    fila.groupBy("acao_recomendada").count().collect()}
assert sum(resumo_operacional.values()) == qtd_candidatos, "Faixas não conciliam com elegíveis"
print("Fila operacional por ação recomendada:", resumo_operacional)
print("Critério: capacidade de 200 é limite operacional, não obrigação de preencher vagas")

# Universo contratado do case; não fixa artificialmente o número de elegíveis.
assert total_ucs == 45000, "Universo diferente das 45.000 UCs contratadas; revisar carga"
QA_EXECUCAO = ID_EXECUCAO


## 6. Autotestes de regra e de fluxo

Casos 100% sintéticos validam as pontuações, retenção por GD/troca/inspeção, arredondamento e a distinção **inspeção prioritária × validação documental**. Estes testes não usam identificadores reais nem publicam tabelas.


In [0]:
# Autotestes V10: 100% sintéticos, sem depender de IDs reais.
# Exercitam o cálculo do score, a queda absoluta condicionada à queda percentual
# e o tratamento de inspeção prévia; arredondamento é apenas campo informativo.
CASOS_TESTE = [
    ("SINTETICO_A", True, .30, 150., 0, 3, 3, False, False, False, False, False, False, False, False),
    ("SINTETICO_B", True, .30, 150., 0, 3, 3, True, False, False, False, False, False, False, False),
    ("SINTETICO_C", False, .30, 150., 0, 1, 1, False, False, False, False, False, False, False, False),
    ("SINTETICO_D", True, .40, 200., 2, 3, 0, False, False, False, False, False, False, False, False),
    ("SINTETICO_E", True, .30, 150., 0, 3, 3, False, False, True, False, False, False, False, True),
    ("SINTETICO_INDUSTRIA_6PCT", True, .06, 700., 0, 3, 3, False, False, False, False, False, False, False, False),
    ("SINTETICO_INSPECAO_ANTIGA", True, .30, 150., 0, 3, 3, False, False, False, False, False, False, False, True),
    ("SINTETICO_ARREDONDAMENTO", True, .30, 150., 0, 3, 3, False, False, False, False, False, True, False, False),
    ("SINTETICO_TROCA", True, .30, 150., 0, 3, 3, False, True, False, False, False, False, False, False),
]
COLUNAS_TESTE = [
    "id_uc", "historico_suficiente", "queda_pct", "queda_kwh_mensal",
    "pares_estimados", "meses_comparaveis", "pares_observados", "gd_apos_base",
    "houve_troca_na_janela", "inspecao_recente", "gd_data_desconhecida",
    "tem_outlier_pareado", "tem_arredondamento_pareado", "tem_sem_leitura_anterior_pareada",
    "ja_inspecionada",
]
sinteticos = spark.createDataFrame(CASOS_TESTE, COLUNAS_TESTE)
sinteticos = calcular_score(sinteticos)
sinteticos = definir_elegibilidade(sinteticos, F.lit(referencia).cast("date"))
resultados = {r["id_uc"]: r for r in sinteticos.select("id_uc", "score", "p_queda_kwh", "status_triagem", "motivos", "fatores_atenuantes").collect()}
assert resultados["SINTETICO_A"]["status_triagem"] == "ELEGIVEL"
assert abs(resultados["SINTETICO_A"]["score"] - 45.0) < 1e-6
assert resultados["SINTETICO_B"]["status_triagem"] == "REVISAR_FATORES_ALTERNATIVOS"
assert abs(resultados["SINTETICO_B"]["score"] - 10.0) < 1e-6
assert resultados["SINTETICO_C"]["status_triagem"] == "HISTORICO_INSUFICIENTE"
assert resultados["SINTETICO_D"]["status_triagem"] == "REVISAR_EVIDENCIA_ESTIMADA"
assert resultados["SINTETICO_E"]["status_triagem"] == "REVISAR_INSPECAO_RECENTE"
assert resultados["SINTETICO_INDUSTRIA_6PCT"]["status_triagem"] == "SEM_EVIDENCIA_MINIMA"
assert "QUEDA_ABSOLUTA_RELEVANTE" not in resultados["SINTETICO_INDUSTRIA_6PCT"]["motivos"]
assert abs(resultados["SINTETICO_INDUSTRIA_6PCT"]["p_queda_kwh"]) < 1e-8, "A3: indústria com 6% de queda pontuou pelo valor absoluto"
assert abs(resultados["SINTETICO_INDUSTRIA_6PCT"]["score"] - 6.0) < 1e-6
assert abs(resultados["SINTETICO_A"]["p_queda_kwh"] - 15.0) < 1e-6
assert resultados["SINTETICO_INSPECAO_ANTIGA"]["status_triagem"] == "ELEGIVEL"
assert "INSPECAO_ANTERIOR_FORA_JANELA" in resultados["SINTETICO_INSPECAO_ANTIGA"]["fatores_atenuantes"]
assert resultados["SINTETICO_ARREDONDAMENTO"]["status_triagem"] == "ELEGIVEL"
assert resultados["SINTETICO_ARREDONDAMENTO"]["score"] == resultados["SINTETICO_A"]["score"]
assert "CONSUMO_CORRIGIDO_OU_INCONSISTENTE" not in resultados["SINTETICO_ARREDONDAMENTO"]["fatores_atenuantes"]
assert resultados["SINTETICO_TROCA"]["status_triagem"] == "REVISAR_FATORES_ALTERNATIVOS"
assert "TROCA_MEDIDOR_NA_JANELA_BASE_ATUAL" in resultados["SINTETICO_TROCA"]["fatores_atenuantes"]
print("Autotestes V10 OK: A1, A2, A3, penalidades e históricos.")

# Regressão A6 final: estimadas recorrentes com queda menor não são descartadas;
# entram somente como validação documental, nunca como ordem direta de campo.
CASOS_FAIXAS = [
    ("TESTE_REVISAO", True, .10, 80., 2, 3, 1, False, False, False, False, False, False, False, False),
    ("TESTE_CAMPO", True, .25, 80., 2, 3, 1, False, False, False, False, False, False, False, False),
    ("TESTE_SEM_SINAL", True, .00, 0., 2, 3, 1, False, False, False, False, False, False, False, False),
]
casos_faixas = spark.createDataFrame(CASOS_FAIXAS, COLUNAS_TESTE)
casos_faixas = definir_elegibilidade(calcular_score(casos_faixas), F.lit(referencia).cast("date"))
faixas_testadas = {r["id_uc"]: r.asDict() for r in casos_faixas.select(
    "id_uc", "status_triagem", "acao_recomendada", "prioridade_operacional", "queda_qualificada").collect()}
assert faixas_testadas["TESTE_REVISAO"]["status_triagem"] == "ELEGIVEL"
assert faixas_testadas["TESTE_REVISAO"]["acao_recomendada"] == "VALIDACAO_DOCUMENTAL"
assert faixas_testadas["TESTE_REVISAO"]["prioridade_operacional"] == 2
assert faixas_testadas["TESTE_CAMPO"]["acao_recomendada"] == "INSPECAO_PRIORITARIA"
assert faixas_testadas["TESTE_CAMPO"]["prioridade_operacional"] == 1
assert faixas_testadas["TESTE_SEM_SINAL"]["status_triagem"] == "BAIXA_PRIORIDADE"
assert faixas_testadas["TESTE_SEM_SINAL"]["acao_recomendada"] == "NAO_APLICAVEL"
print("Autotestes finais OK: A1–A3, score, retenções e ação recomendada em duas faixas")

AUTOTESTES_EXECUCAO = ID_EXECUCAO


## Publicação e recuperação

Valida o contrato completo antes e após a escrita, incluindo elegíveis, ranking,
tipos, ações, score e metadados. Registra parâmetros e execução em `userMetadata`
de cada commit Delta. `PUBLICACAO_EFETIVADA` só se torna verdadeiro após a conferência.

Em falha, tenta restaurar apenas destinos tentados cujo HEAD ainda seja o commit
isolado desta execução. Confere conteúdo restaurado. Tabela sem versão anterior,
concorrência ou falha no RESTORE exigem recuperação manual; nenhum DROP é automático.
Duas tabelas não formam transação atômica. Falha do processo pode impedir até a tentativa
de recuperação: consultar histórico e metadados antes de retomar o consumo.


In [0]:
# Delta garante atomicidade por tabela. Operar com um único publicador.
# max_concurrent_runs=1 no Job e nenhuma execução manual concorrente.
PUBLICACAO_EFETIVADA = False

def exigir_condicao(condicao, mensagem):
    if not condicao:
        raise RuntimeError(mensagem)

def conferir_igualdade(actual, esperado, rotulo):
    exigir_condicao(actual.columns == esperado.columns, f"{rotulo}: colunas divergentes")
    exigir_condicao(
        [(f.name, f.dataType) for f in actual.schema] ==
        [(f.name, f.dataType) for f in esperado.schema], f"{rotulo}: tipos divergentes")
    exigir_condicao(not actual.exceptAll(esperado).limit(1).count() and
                     not esperado.exceptAll(actual).limit(1).count(),
                     f"{rotulo}: conteúdo divergente (comparação integral)")

def validar_contrato_publicacao(auditoria, candidatas):
    exigir_condicao(auditoria.count() == total_ucs == 45000, "Auditoria incompleta")
    exigir_condicao(candidatas.count() == qtd_candidatos, "Total da fila divergente")
    for rotulo, df in (("auditoria", auditoria), ("fila", candidatas)):
        exigir_condicao(not df.where(F.col("id_uc").isNull()).limit(1).count(), f"{rotulo}: UC nula")
        exigir_condicao(not df.groupBy("id_uc").count().where(F.col("count") > 1).limit(1).count(),
                         f"{rotulo}: UC duplicada")
        invalidos = (~F.col("id_execucao").eqNullSafe(F.lit(ID_EXECUCAO)) |
                     ~F.col("versao_regra").eqNullSafe(F.lit(VERSAO_REGRA)) |
                     ~F.col("versoes_silver_json").eqNullSafe(F.lit(VERSOES_FONTES_JSON)) |
                     F.col("executado_em_utc").isNull() |
                     F.col("status_triagem").isNull() |
                     F.col("score").isNull() | F.isnan("score") |
                     (F.col("score") < 0) | (F.col("score") > 100))
        exigir_condicao(not df.where(invalidos).limit(1).count(), f"{rotulo}: metadados/score inválidos")
        acao = (F.when(F.col("status_triagem") != "ELEGIVEL", F.lit("NAO_APLICAVEL"))
                .when(F.col("queda_qualificada"), F.lit("INSPECAO_PRIORITARIA"))
                .otherwise(F.lit("VALIDACAO_DOCUMENTAL")))
        prioridade = (F.when(acao == "INSPECAO_PRIORITARIA", 1)
                      .when(acao == "VALIDACAO_DOCUMENTAL", 2).otherwise(9))
        exigir_condicao(not df.where(~F.col("acao_recomendada").eqNullSafe(acao) |
                                    ~F.col("prioridade_operacional").eqNullSafe(prioridade) |
                                    F.col("queda_qualificada").isNull()).limit(1).count(),
                         f"{rotulo}: ação/prioridade incoerente")
    exigir_condicao(not auditoria.select("id_uc").exceptAll(uc.select("id_uc")).limit(1).count(),
                     "Auditoria contém UC fora do cadastro")
    elegiveis = auditoria.where(F.col("status_triagem") == "ELEGIVEL")
    esperada = elegiveis.withColumn("posicao_fila", F.row_number().over(JANELA_RANKING))
    conferir_igualdade(candidatas, esperada, "Fila versus elegíveis e ranking")

def estado_delta(nome):
    if not spark.catalog.tableExists(nome):
        return None
    linha = spark.sql(f"DESCRIBE HISTORY {nome} LIMIT 1").select("version", "userMetadata").first()
    exigir_condicao(linha is not None, f"Sem histórico Delta: {nome}")
    return {"version": int(linha["version"]), "userMetadata": linha["userMetadata"]}

def commit_proprio_isolado(antes, atual, marcador):
    # Também reconhece commit efetivado quando saveAsTable lança erro após o commit.
    return (atual is not None and atual["userMetadata"] == marcador and
            atual["version"] == (0 if antes is None else antes["version"] + 1))

parametros_agora = ler_parametros_efetivos()
exigir_condicao(parametros_agora == PARAMETROS_EFETIVOS,
                 "Widgets mudaram após o cálculo. Configure-os e execute Run All novamente.")
publicar_agora, modo_agora = decidir_publicacao(parametros_agora)
exigir_condicao(publicar_agora == PUBLICAR_GOLD, "Estado de publicação alterado; execute Run All")
print("Pré-publicação:", parametros_agora, "|", modo_agora)
conferir_origens_inalteradas()
if publicar_agora:
    exigir_condicao(QA_EXECUCAO == ID_EXECUCAO and AUTOTESTES_EXECUCAO == ID_EXECUCAO,
                     "QA/autotestes não concluídos nesta execução. Execute Run All.")
    exigir_condicao(not PUBLICACAO_TENTADA, "Publicação já tentada; execute Run All para nova tentativa")
    validar_contrato_publicacao(universo_atual, fila)
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD}")
    nome_gold = f"{GOLD}.tb_fila_perdas"
    nome_auditoria = f"{GOLD}.tb_fila_perdas_auditoria"
    destinos = [(nome_auditoria, universo_atual), (nome_gold, fila)]
    anteriores = {nome: estado_delta(nome) for nome, _ in destinos}
    marcador_commit = json.dumps({"id_execucao": ID_EXECUCAO, "versao_regra": VERSAO_REGRA,
                                  "parametros": PARAMETROS_EFETIVOS}, sort_keys=True)
    print("Versões anteriores:", anteriores)
    tentadas = []
    gravadas = {}
    PUBLICACAO_TENTADA = True
    try:
        for nome, df in destinos:
            exigir_condicao(estado_delta(nome) == anteriores[nome],
                             f"Destino mudou antes da escrita: {nome}. Suspender concorrência.")
            tentadas.append(nome)
            (df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
                .option("userMetadata", marcador_commit).saveAsTable(nome))
            atual = estado_delta(nome)
            exigir_condicao(commit_proprio_isolado(anteriores[nome], atual, marcador_commit),
                             f"Commit inesperado ou concorrência: {nome}")
            gravadas[nome] = atual
        # Time travel fixa a dupla conferida; HEAD é checado novamente ao terminar.
        aud = spark.sql(f"SELECT * FROM {nome_auditoria} VERSION AS OF {gravadas[nome_auditoria]['version']}")
        fil = spark.sql(f"SELECT * FROM {nome_gold} VERSION AS OF {gravadas[nome_gold]['version']}")
        validar_contrato_publicacao(aud, fil)
        conferir_igualdade(aud, universo_atual, "Auditoria publicada versus calculada")
        conferir_igualdade(fil, fila, "Fila publicada versus calculada")
        conferir_origens_inalteradas()
        exigir_condicao(all(estado_delta(nome) == gravadas[nome] for nome, _ in destinos),
                         "HEAD mudou durante a verificação. Publicação não confirmada.")
    except Exception as erro_publicacao:
        falhas_restauracao = []
        for nome in reversed(tentadas):
            try:
                antes, atual = anteriores[nome], estado_delta(nome)
                if atual == antes:
                    print(f"Sem alteração a restaurar: {nome}")
                    continue
                exigir_condicao(commit_proprio_isolado(antes, atual, marcador_commit),
                                 f"{nome}: HEAD não é commit isolado desta execução; recuperação manual")
                # Sem versão anterior, não fazer DROP automático de tabela.
                exigir_condicao(antes is not None,
                                 f"{nome}: primeira criação sem backup; recuperação manual")
                historico = spark.sql(f"SELECT * FROM {nome} VERSION AS OF {antes['version']}")
                spark.sql(f"RESTORE TABLE {nome} TO VERSION AS OF {antes['version']}")
                restaurado = estado_delta(nome)
                exigir_condicao(restaurado['version'] == atual['version'] + 1,
                                 f"{nome}: versão inesperada após RESTORE")
                conferir_igualdade(spark.sql(
                    f"SELECT * FROM {nome} VERSION AS OF {restaurado['version']}"), historico,
                    f"RESTORE {nome}")
                exigir_condicao(estado_delta(nome) == restaurado, f"{nome}: HEAD mudou após RESTORE")
                print(f"RESTORE verificado: {nome} -> conteúdo da versão {antes['version']}")
            except Exception as erro_restore:
                falhas_restauracao.append((nome, str(erro_restore)))
        print("Pendências de recuperação:", falhas_restauracao)
        raise RuntimeError("Publicação incompleta. Não liberar consumidores. Conferir as duas tabelas "
                           "e resolver pendências; RESTORE não constitui transação conjunta.") from erro_publicacao
    PUBLICACAO_EFETIVADA = True
    print("PUBLICAÇÃO CONFIRMADA:", {nome: estado['version'] for nome, estado in gravadas.items()})
    print(f"id_execucao={ID_EXECUCAO}; versao_regra={VERSAO_REGRA}; auditoria={total_ucs}; fila={qtd_candidatos}")
else:
    print(f"MODO {modo_agora}. Nenhuma tabela Gold foi alterada.")
    print(f"V10 calculada em memória: {qtd_candidatos} candidatas; id_execucao={ID_EXECUCAO}")
    nome_gold = f"{GOLD}.tb_fila_perdas"
    if spark.catalog.tableExists(nome_gold):
        anterior = spark.table(nome_gold)
        print("Publicação anterior:", anterior.count(), "elegíveis; versões=",
              [r[0] for r in anterior.select("versao_regra").distinct().limit(5).collect()])


## 8. Contrato de integração com o portal

A fila contém duas ações distintas. Uma UC em `VALIDACAO_DOCUMENTAL` **não deve virar ordem automática de campo**. Na interface, apresentar filtro por `acao_recomendada`, score, classe, município, motivos e estimativa de kWh, preservando a ordenação `posicao_fila`.

A decisão final da visita e o registro dos resultados pertencem ao workflow do portal. `kwh_estimado` é um indicador auxiliar de triagem, não medição verificada de perda. O backtest está apenas na exploração e possui viés de seleção do histórico de inspeções.

### Verificações ao concluir a execução
- QA e autotestes aprovados;
- soma das duas ações igual ao total elegível;
- `inspecao_recente` ausente entre elegíveis;
- leituras negativas inválidas não entram nas médias;
- as cinco versões Silver são registradas e não mudam durante a execução;
- após publicar, ambas as tabelas apresentam a mesma `id_execucao`.


In [0]:
print("Gold FINAL V10:")
print(f"Competência={referencia} | versão={VERSAO_REGRA} | id_execucao={ID_EXECUCAO}")
print(f"UCs auditadas={total_ucs} | oportunidades na fila={qtd_candidatos} | ação por faixa={resumo_operacional}")
print(f"Publicação efetivada={PUBLICACAO_EFETIVADA}; fontes Silver={VERSOES_FONTES_JSON}")
